In [16]:

%pip -q install duckdb==1.5.6 pandas dbt-core==1.9.4 dbt-duckdb==1.9.4

import sys, json, shutil, subprocess
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal

import duckdb
import pandas as pd

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Data_Lakehouse_Project")
else:
    ROOT = Path.cwd()

if not ROOT.is_dir():
    raise FileNotFoundError(f"Không tìm thấy dự án: {ROOT}")

DB_CANDIDATES = [
    ROOT / "database/credit_lakehouse.duckdb",
    ROOT / "data/lakehouse_db/credit_lakehouse.duckdb",
]
DB_PATH = next((p for p in DB_CANDIDATES if p.exists()), None)

if DB_PATH is None:
    raise FileNotFoundError("Không tìm thấy credit_lakehouse.duckdb")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = ROOT / "exports" / "pipeline_reconciliation_validation" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

TMP = Path("/content/task41_duckdb_tmp") if IN_COLAB else ROOT / "tmp/task41_duckdb"
TMP.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute("SET memory_limit='4GB'")
con.execute("SET threads=4")
con.execute("SET temp_directory=?", [str(TMP)])

print("Run ID:", RUN_ID)
print("Database:", DB_PATH)
print("Evidence:", OUT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Run ID: 20261106T191837Z
Database: /content/drive/MyDrive/Data_Lakehouse_Project/database/credit_lakehouse.duckdb
Evidence: /content/drive/MyDrive/Data_Lakehouse_Project/exports/pipeline_reconciliation_validation/20261106T191837Z


In [17]:

records = []

def record(test_id, group, status, observed, note):
    item = {
        "test_id": test_id,
        "group": group,
        "status": status,
        "observed": json.dumps(observed, ensure_ascii=False, default=str),
        "note": note,
    }
    records.append(item)
    print(f"[{status}] {test_id}: {item['observed']}")
    return item

def audit(test_id, group, query, judge, note):
    try:
        cur = con.execute(query)
        values = cur.fetchone()
        columns = [col[0] for col in cur.description]
        observed = dict(zip(columns, values))
        status = judge(observed)
        if isinstance(status, bool):
            status = "PASS" if status else "FAIL"
        record(test_id, group, status, observed, note)
    except Exception as exc:
        record(test_id, group, "ERROR", {"error": str(exc)}, note)

source_to_staging = {
    "application_train": "stg_application_train",
    "application_test": "stg_application_test",
    "bureau": "stg_bureau",
    "bureau_balance": "stg_bureau_balance",
    "previous_application": "stg_previous_application",
    "pos_cash_balance": "stg_pos_cash_balance",
    "credit_card_balance": "stg_credit_card_balance",
    "installments_payments": "stg_installments_payments",
}

relations = con.execute("""
    SELECT table_schema || '.' || table_name AS relation
    FROM information_schema.tables
    WHERE table_schema IN ('bronze', 'silver', 'gold')
""").fetchdf()

available = set(relations["relation"])

required = (
    [f"bronze.{s}" for s in source_to_staging]
    + [f"silver.{s}" for s in source_to_staging.values()]
    + ["silver.int_application_train_prep",
       "gold.fct_application_train",
       "gold.feature_mart_application_train"]
)

missing = sorted(set(required) - available)
record(
    "T41_01", "Structure",
    "PASS" if not missing else "FAIL",
    {"relations_found": len(available), "missing_relations": missing},
    "Kiểm tra sự hiện diện của 19 bảng/view theo pipeline hiện có."
)

for source, stage in source_to_staging.items():
    audit(
        f"T41_ROWS_{source}", "Bronze-Silver",
        f"""
        SELECT
          (SELECT COUNT(*) FROM bronze.{source}) AS bronze_rows,
          (SELECT COUNT(*) FROM silver.{stage}) AS silver_rows
        """,
        lambda r: r["bronze_rows"] == r["silver_rows"],
        "Kiểm tra bảo toàn số bản ghi giữa Bronze và Silver."
    )


[PASS] T41_01: {"relations_found": 19, "missing_relations": []}
[PASS] T41_ROWS_application_train: {"bronze_rows": 307511, "silver_rows": 307511}
[PASS] T41_ROWS_application_test: {"bronze_rows": 48744, "silver_rows": 48744}
[PASS] T41_ROWS_bureau: {"bronze_rows": 1716428, "silver_rows": 1716428}
[PASS] T41_ROWS_bureau_balance: {"bronze_rows": 27299925, "silver_rows": 27299925}
[PASS] T41_ROWS_previous_application: {"bronze_rows": 1670214, "silver_rows": 1670214}
[PASS] T41_ROWS_pos_cash_balance: {"bronze_rows": 10001358, "silver_rows": 10001358}
[PASS] T41_ROWS_credit_card_balance: {"bronze_rows": 3840312, "silver_rows": 3840312}
[PASS] T41_ROWS_installments_payments: {"bronze_rows": 13605401, "silver_rows": 13605401}


In [18]:

audit(
    "T41_10", "Bronze-Gold",
    """
    WITH a AS (
        SELECT
          TRY_CAST(SK_ID_CURR AS BIGINT) AS k,
          TRY_CAST(TARGET AS INTEGER) AS t,
          TRY_CAST(AMT_CREDIT AS DECIMAL(20,2)) AS amount
        FROM bronze.application_train
    ),
    m AS (
        SELECT
          TRY_CAST(sk_id_curr AS BIGINT) AS k,
          TRY_CAST(target AS INTEGER) AS t,
          TRY_CAST(amt_credit AS DECIMAL(20,2)) AS amount
        FROM gold.feature_mart_application_train
    )
    SELECT
      (SELECT COUNT(*) FROM a) AS source_rows,
      (SELECT COUNT(*) FROM m) AS mart_rows,
      (SELECT COUNT(DISTINCT k) FROM a) AS source_unique,
      (SELECT COUNT(DISTINCT k) FROM m) AS mart_unique,
      (SELECT COUNT(*) FROM a WHERE k IS NULL) AS source_null,
      (SELECT COUNT(*) FROM m WHERE k IS NULL) AS mart_null,
      (SELECT COUNT(*) FROM
        (SELECT k FROM a EXCEPT SELECT k FROM m)
      ) AS missing_in_mart,
      (SELECT COUNT(*) FROM
        (SELECT k FROM m EXCEPT SELECT k FROM a)
      ) AS extra_in_mart,
      (SELECT COUNT(*) FROM a JOIN m USING (k)
       WHERE a.t IS DISTINCT FROM m.t
          OR a.amount IS DISTINCT FROM m.amount
      ) AS value_mismatches
    """,
    lambda r: (
        r["source_rows"] == r["mart_rows"]
        == r["source_unique"] == r["mart_unique"]
        and all(r[k] == 0 for k in [
            "source_null", "mart_null",
            "missing_in_mart", "extra_in_mart", "value_mismatches"
        ])
    ),
    "Kiểm tra fan-out, tập khóa hai chiều, TARGET và AMT_CREDIT."
)

audit(
    "T41_11", "Gold-Gold",
    """
    WITH f AS (
      SELECT sk_id_curr AS k FROM gold.fct_application_train
    ),
    m AS (
      SELECT sk_id_curr AS k FROM gold.feature_mart_application_train
    )
    SELECT
      (SELECT COUNT(*) FROM f) AS fact_rows,
      (SELECT COUNT(*) FROM m) AS mart_rows,
      (SELECT COUNT(DISTINCT k) FROM f) AS fact_unique,
      (SELECT COUNT(DISTINCT k) FROM m) AS mart_unique,
      (SELECT COUNT(*) FROM (SELECT k FROM f EXCEPT SELECT k FROM m))
        AS missing_in_mart,
      (SELECT COUNT(*) FROM (SELECT k FROM m EXCEPT SELECT k FROM f))
        AS extra_in_mart
    """,
    lambda r: (
        r["fact_rows"] == r["mart_rows"]
        == r["fact_unique"] == r["mart_unique"]
        and r["missing_in_mart"] == 0
        and r["extra_in_mart"] == 0
    ),
    "Đảm bảo Fact và Feature Mart cùng grain khách hàng."
)

audit(
    "T41_12", "Business KPI",
    """
    SELECT
      (SELECT COUNT(*) FROM bronze.application_train
       WHERE TRY_CAST(TARGET AS INTEGER)=1) AS source_target_1,
      (SELECT COUNT(*) FROM gold.feature_mart_application_train
       WHERE target=1) AS mart_target_1
    """,
    lambda r: r["source_target_1"] == r["mart_target_1"],
    "Đối chiếu số hồ sơ TARGET=1, chỉ trên tập Train."
)

audit(
    "T41_13", "Derived Features",
    """
    SELECT COUNT(*) AS inconsistent_rows
    FROM gold.feature_mart_application_train
    WHERE credit_to_income IS DISTINCT FROM
      CASE WHEN amt_income_total > 0
           THEN amt_credit / amt_income_total END
       OR annuity_to_income IS DISTINCT FROM
      CASE WHEN amt_income_total > 0
           THEN amt_annuity / amt_income_total END
    """,
    lambda r: r["inconsistent_rows"] == 0,
    "Kiểm tra hai tỷ số tài chính theo logic dbt hiện hành."
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[PASS] T41_10: {"source_rows": 307511, "mart_rows": 307511, "source_unique": 307511, "mart_unique": 307511, "source_null": 0, "mart_null": 0, "missing_in_mart": 0, "extra_in_mart": 0, "value_mismatches": 0}
[PASS] T41_11: {"fact_rows": 307511, "mart_rows": 307511, "fact_unique": 307511, "mart_unique": 307511, "missing_in_mart": 0, "extra_in_mart": 0}
[PASS] T41_12: {"source_target_1": 24825, "mart_target_1": 24825}
[PASS] T41_13: {"inconsistent_rows": 0}


In [19]:

audit(
    "T41_20", "Payment Reconciliation",
    """
    WITH src AS (
      SELECT
        COUNT(*) AS n,
        COUNT(TRY_CAST(AMT_PAYMENT AS DECIMAL(20,2))) AS valid_n,
        SUM(TRY_CAST(AMT_PAYMENT AS DECIMAL(20,2))) AS total
      FROM bronze.installments_payments
    ),
    tgt AS (
      SELECT
        COUNT(*) AS n,
        COUNT(TRY_CAST(normalized_amt_payment AS DECIMAL(20,2)))
          AS valid_n,
        SUM(TRY_CAST(normalized_amt_payment AS DECIMAL(20,2)))
          AS total
      FROM silver.stg_installments_payments
    )
    SELECT
      src.n AS source_rows, tgt.n AS silver_rows,
      src.valid_n AS source_valid, tgt.valid_n AS silver_valid,
      src.n-src.valid_n AS source_missing,
      tgt.n-tgt.valid_n AS silver_missing,
      src.total AS source_total,
      tgt.total AS silver_total,
      ABS(COALESCE(src.total,0)-COALESCE(tgt.total,0))
        AS amount_difference
    FROM src CROSS JOIN tgt
    """,
    lambda r: (
        r["source_rows"] == r["silver_rows"]
        and r["source_valid"] == r["silver_valid"]
        and r["source_missing"] == r["silver_missing"]
        and r["amount_difference"] <= Decimal("0.01")
    ),
    "Đối soát số dòng, NULL và tổng thanh toán bằng DECIMAL."
)

audit(
    "T41_21", "Installment Grain",
    """
    WITH periods AS (
      SELECT
        SK_ID_PREV,
        NUM_INSTALMENT_VERSION,
        NUM_INSTALMENT_NUMBER,
        COUNT(DISTINCT TRY_CAST(
          AMT_INSTALMENT AS DECIMAL(20,4)
        )) AS due_variants,
        COUNT(*) AS transaction_count
      FROM bronze.installments_payments
      GROUP BY 1,2,3
    )
    SELECT
      COUNT(*) AS installment_periods,
      COUNT(*) FILTER (WHERE due_variants > 1)
        AS conflicting_periods,
      COUNT(*) FILTER (WHERE transaction_count > 1)
        AS multi_transaction_periods
    FROM periods
    """,
    lambda r: "PASS" if r["conflicting_periods"] == 0 else "REVIEW",
    "Kiểm tra grain kỳ hạn và các kỳ có nhiều nghĩa vụ thanh toán."
)

for table in [
    "stg_pos_cash_balance",
    "stg_installments_payments",
    "stg_credit_card_balance",
]:
    audit(
        f"T41_REL_{table}", "Relationship Coverage",
        f"""
        WITH customers AS (
          SELECT normalized_sk_id_curr AS k
          FROM silver.stg_application_train
          UNION
          SELECT normalized_sk_id_curr AS k
          FROM silver.stg_application_test
        ),
        previous_keys AS (
          SELECT normalized_sk_id_prev AS p
          FROM silver.stg_previous_application
        )
        SELECT
          COUNT(*) AS source_rows,
          COUNT(*) FILTER (
            WHERE p.p IS NULL
          ) AS unmatched_previous,
          COUNT(*) FILTER (
            WHERE c.k IS NULL
          ) AS unmatched_customer
        FROM silver.{table} s
        LEFT JOIN previous_keys p
          ON s.normalized_sk_id_prev=p.p
        LEFT JOIN customers c
          ON s.normalized_sk_id_curr=c.k
        """,
        lambda r: "PASS" if r["unmatched_customer"] == 0 else "REVIEW",
        "SK_ID_PREV không khớp được ghi nhận riêng; không tự động loại bản ghi."
    )

audit(
    "T41_REL_BUREAU", "Relationship Coverage",
    """
    WITH bb AS (
      SELECT DISTINCT normalized_sk_id_bureau AS k
      FROM silver.stg_bureau_balance
      WHERE normalized_sk_id_bureau IS NOT NULL
    ),
    b AS (
      SELECT normalized_sk_id_bureau AS k
      FROM silver.stg_bureau
    )
    SELECT COUNT(*) AS unmatched_distinct_bureau_keys
    FROM bb LEFT JOIN b USING(k)
    WHERE b.k IS NULL
    """,
    lambda r: "INFO",
    "Thống kê khóa Bureau Balance chưa nối được; không mặc định là lỗi pipeline."
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[PASS] T41_20: {"source_rows": 13605401, "silver_rows": 13605401, "source_valid": 13602496, "silver_valid": 13602496, "source_missing": 2905, "silver_missing": 2905, "source_total": "234482890293.03", "silver_total": "234482890293.03", "amount_difference": "0.00"}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[PASS] T41_21: {"installment_periods": 12951918, "conflicting_periods": 0, "multi_transaction_periods": 640905}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[PASS] T41_REL_stg_pos_cash_balance: {"source_rows": 10001358, "unmatched_previous": 340561, "unmatched_customer": 0}


... (2 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[PASS] T41_REL_stg_credit_card_balance: {"source_rows": 3840312, "unmatched_previous": 1082816, "unmatched_customer": 0}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[INFO] T41_REL_BUREAU: {"unmatched_distinct_bureau_keys": 43041}


In [20]:

# Giải phóng file database trước khi chạy dbt.
con.close()

DBT_DIR = ROOT / "dbt"
PROFILE_DIR = OUT / "dbt_profiles"
TARGET_DIR = OUT / "dbt_target"
PROFILE_DIR.mkdir(parents=True, exist_ok=True)
TARGET_DIR.mkdir(parents=True, exist_ok=True)

profiles_yaml = (
    "credit_lakehouse:\n"
    "  target: local\n"
    "  outputs:\n"
    "    local:\n"
    "      type: duckdb\n"
    f"      path: {json.dumps(str(DB_PATH))}\n"
    "      threads: 4\n"
)
(PROFILE_DIR / "profiles.yml").write_text(
    profiles_yaml, encoding="utf-8"
)

if DBT_DIR.is_dir() and (DBT_DIR / "dbt_project.yml").exists():
    dbt_command = [
        shutil.which("dbt") or "dbt",
        "test",
        "--project-dir", str(DBT_DIR),
        "--profiles-dir", str(PROFILE_DIR),
        "--target-path", str(TARGET_DIR),
    ]
    proc = subprocess.run(
        dbt_command,
        capture_output=True,
        text=True,
        check=False,
    )

    (OUT / "dbt_test_console.log").write_text(
        proc.stdout + "\n" + proc.stderr,
        encoding="utf-8",
    )

    run_results_path = TARGET_DIR / "run_results.json"
    if run_results_path.exists():
        artifact = json.loads(run_results_path.read_text(encoding="utf-8"))
        results = [
            r for r in artifact.get("results", [])
            if r.get("unique_id", "").startswith("test.")
        ]
        statuses = {}
        for r in results:
            s = r.get("status", "unknown")
            statuses[s] = statuses.get(s, 0) + 1
        dbt_ok = (
            proc.returncode == 0
            and len(results) > 0
            and statuses.get("pass", 0) == len(results)
        )
        record(
            "T41_DBT", "dbt Re-test",
            "PASS" if dbt_ok else "FAIL",
            {"tests": len(results), "statuses": statuses,
             "returncode": proc.returncode},
            "Trạng thái chạy lại dbt test trên dự án hiện có."
        )
    else:
        record(
            "T41_DBT", "dbt Re-test", "ERROR",
            {"returncode": proc.returncode},
            "Không tìm thấy run_results.json từ lần kiểm thử mới."
        )
else:
    record(
        "T41_DBT", "dbt Re-test", "NOT_RUN",
        {"dbt_directory": str(DBT_DIR)},
        "Thiếu cấu trúc dự án dbt trong Google Drive."
    )

df_results = pd.DataFrame(records)
csv_path = OUT / "pipeline_validation_results.csv"
df_results.to_csv(csv_path, index=False, encoding="utf-8-sig")

status_counts = df_results["status"].value_counts().to_dict()
blocking = any(
    status_counts.get(s, 0) > 0
    for s in ["FAIL", "ERROR", "REVIEW", "NOT_RUN"]
)

summary = {
    "task": "Rà soát pipeline và kiểm chứng tính nhất quán dữ liệu",
    "run_id": RUN_ID,
    "database": str(DB_PATH),
    "check_count": len(records),
    "status_counts": status_counts,
    "final_state": "NEEDS_REVIEW" if blocking else "PASS",
    "results_file": str(csv_path),
    "dbt_log_file": str(OUT / "dbt_test_console.log"),
}

summary_path = OUT / "pipeline_validation_summary.json"
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display(df_results[["test_id", "group", "status", "note"]])
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Đã xuất:", csv_path)
print("Đã xuất:", summary_path)


[PASS] T41_DBT: {"tests": 26, "statuses": {"pass": 26}, "returncode": 0}


,test_id,group,status,note
0,T41_01,Structure,PASS,Kiểm tra sự hiện diện của 19 bảng/view theo pi...
1,T41_ROWS_application_train,Bronze-Silver,PASS,Kiểm tra bảo toàn số bản ghi giữa Bronze và Si...
2,T41_ROWS_application_test,Bronze-Silver,PASS,Kiểm tra bảo toàn số bản ghi giữa Bronze và Si...
3,T41_ROWS_bureau,Bronze-Silver,PASS,Kiểm tra bảo toàn số bản ghi giữa Bronze và Si...
4,T41_ROWS_bureau_balance,Bronze-Silver,PASS,Kiểm tra bảo toàn số bản ghi giữa Bronze và Si...
5,T41_ROWS_previous_application,Bronze-Silver,PASS,Kiểm tra bảo toàn số bản ghi giữa Bronze và Si...
6,T41_ROWS_pos_cash_balance,Bronze-Silver,PASS,Kiểm tra bảo toàn số bản ghi giữa Bronze và Si...
7,T41_ROWS_credit_card_balance,Bronze-Silver,PASS,Kiểm tra bảo toàn số bản ghi giữa Bronze và Si...
8,T41_ROWS_installments_payments,Bronze-Silver,PASS,Kiểm tra bảo toàn số bản ghi giữa Bronze và Si...
9,T41_10,Bronze-Gold,PASS,"Kiểm tra fan-out, tập khóa hai chiều, TARGET v..."


{
  "task": "Rà soát pipeline và kiểm chứng tính nhất quán dữ liệu",
  "run_id": "20261106T191837Z",
  "database": "/content/drive/MyDrive/Data_Lakehouse_Project/database/credit_lakehouse.duckdb",
  "check_count": 20,
  "status_counts": {
    "PASS": 19,
    "INFO": 1
  },
  "final_state": "PASS",
  "results_file": "/content/drive/MyDrive/Data_Lakehouse_Project/exports/pipeline_reconciliation_validation/20261106T191837Z/pipeline_validation_results.csv",
  "dbt_log_file": "/content/drive/MyDrive/Data_Lakehouse_Project/exports/pipeline_reconciliation_validation/20261106T191837Z/dbt_test_console.log"
}
Đã xuất: /content/drive/MyDrive/Data_Lakehouse_Project/exports/pipeline_reconciliation_validation/20261106T191837Z/pipeline_validation_results.csv
Đã xuất: /content/drive/MyDrive/Data_Lakehouse_Project/exports/pipeline_reconciliation_validation/20261106T191837Z/pipeline_validation_summary.json
